In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

from sklearn.metrics import accuracy_score, classification_report

In [2]:
# Step 2: Load the Breast Cancer dataset
data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset Loaded Successfully")
print("Dataset Shape:", X.shape)
print("Number of Features:", X.shape[1])

Dataset Loaded Successfully
Dataset Shape: (569, 30)
Number of Features: 30


In [3]:
# Step 3: Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)


Training Data: (455, 30)
Testing Data: (114, 30)


In [4]:
# Step 4: Define cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [5]:
# Step 5: Create machine learning models
models = {
    "SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", SVC())
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(random_state=42))
    ]),

    "AdaBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", AdaBoostClassifier(random_state=42))
    ])
}

In [6]:
# Step 6: Perform Cross Validation
print("\n========== CROSS VALIDATION ==========")

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    print("\nModel:", name)
    print("CV Scores:", scores)
    print("Mean CV Accuracy:", scores.mean())


========== CROSS VALIDATION ==========

Model: SVM
CV Scores: [0.94505495 0.97802198 0.96703297 0.96703297 0.98901099]
Mean CV Accuracy: 0.9692307692307693

Model: Random Forest
CV Scores: [0.96703297 0.95604396 0.93406593 0.96703297 0.98901099]
Mean CV Accuracy: 0.9626373626373628

Model: AdaBoost
CV Scores: [0.98901099 0.94505495 0.94505495 0.96703297 0.98901099]
Mean CV Accuracy: 0.9670329670329672


In [7]:
# Step 7: Define hyperparameter grids
param_grids = {
    "SVM": {
        "classifier__C": [0.1, 1, 10],
        "classifier__kernel": ["linear", "rbf"]
    },

    "Random Forest": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [None, 5, 10]
    },

    "AdaBoost": {
        "classifier__n_estimators": [50, 100],
        "classifier__learning_rate": [0.5, 1.0]
    }
}

In [8]:
# Step 8: Perform Grid Search and evaluate models
results = []

print("\n========== GRID SEARCH ==========")

for name, model in models.items():

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    # Train using Grid Search
    grid.fit(X_train, y_train)

    # Best model
    best_model = grid.best_estimator_

    # Predict on test data
    y_pred = best_model.predict(X_test)

    # Calculate test accuracy
    test_accuracy = accuracy_score(y_test, y_pred)

    print("\nModel:", name)
    print("Best Parameters:", grid.best_params_)
    print("Best CV Accuracy:", grid.best_score_)
    print("Test Accuracy:", test_accuracy)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=data.target_names
    ))

    results.append({
        "Model": name,
        "Best CV Accuracy": grid.best_score_,
        "Test Accuracy": test_accuracy,
        "Best Parameters": grid.best_params_
    })


========== GRID SEARCH ==========

Model: SVM
Best Parameters: {'classifier__C': 0.1, 'classifier__kernel': 'linear'}
Best CV Accuracy: 0.9758241758241759
Test Accuracy: 0.9824561403508771

Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


Model: Random Forest
Best Parameters: {'classifier__max_depth': None, 'classifier__n_estimators': 100}
Best CV Accuracy: 0.9626373626373628
Test Accuracy: 0.956140350877193

Classification Report:
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95

In [9]:
# Step 9: Compare all models
print("\n========== MODEL COMPARISON ==========")

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))


========== MODEL COMPARISON ==========
        Model  Best CV Accuracy  Test Accuracy                                                     Best Parameters
          SVM          0.975824       0.982456              {'classifier__C': 0.1, 'classifier__kernel': 'linear'}
Random Forest          0.962637       0.956140    {'classifier__max_depth': None, 'classifier__n_estimators': 100}
     AdaBoost          0.980220       0.956140 {'classifier__learning_rate': 1.0, 'classifier__n_estimators': 100}


In [10]:
# Step 10: Find the best performing model
best_result = result_df.loc[
    result_df["Test Accuracy"].idxmax()
]

print("\nBest Performing Model:")
print(best_result["Model"])

print("Best Test Accuracy:", best_result["Test Accuracy"])


Best Performing Model:
SVM
Best Test Accuracy: 0.9824561403508771
